# 06 — Pipeline Evaluation & Executive Summary
End-to-end summary of the Retailers AI Pricing system.

This notebook consolidates all outputs into a single readable report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
PROCESSED = Path('../data/processed')
print('Imports OK')

## 1. Load all pipeline outputs

In [ ]:
tx       = pd.read_parquet(PROCESSED / 'transactions.parquet')
products = pd.read_parquet(PROCESSED / 'product_features.parquet')
monthly  = pd.read_parquet(PROCESSED / 'monthly_features.parquet')
demand   = pd.read_parquet(PROCESSED / 'demand_predictions.parquet')
elast    = pd.read_parquet(PROCESSED / 'elasticity.parquet')
recs     = pd.read_parquet(PROCESSED / 'price_recommendations.parquet')

print('=== Pipeline outputs loaded ===')
print(f'Transactions        : {len(tx):,} rows')
print(f'Products            : {len(products):,} unique SKUs')
print(f'Monthly demand rows : {len(monthly):,}')
print(f'Demand predictions  : {len(demand):,}')
print(f'Elasticity estimates: {len(elast):,} products')
print(f'Price recommendations: {len(recs):,} products')

## 2. Dataset summary

In [ ]:
total_revenue = tx['Revenue'].sum()
date_min = tx['InvoiceDate'].min().strftime('%b %Y')
date_max = tx['InvoiceDate'].max().strftime('%b %Y')
n_countries = tx['Country'].nunique()
n_customers = tx['CustomerID'].nunique()
top_country = tx.groupby('Country')['Revenue'].sum().idxmax()
top_country_pct = tx.groupby('Country')['Revenue'].sum().max() / total_revenue * 100

print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Period              : {date_min} — {date_max}')
print(f'Total transactions  : {len(tx):,}')
print(f'Total revenue       : £{total_revenue:,.0f}')
print(f'Unique products     : {len(products):,}')
print(f'Unique customers    : {n_customers:,}')
print(f'Countries           : {n_countries}')
print(f'Dominant market     : {top_country} ({top_country_pct:.1f}% of revenue)')
print(f'Avg order value     : £{total_revenue / tx["InvoiceNo"].nunique():.2f}')

## 3. Model performance summary

In [ ]:
# Demand model performance on validation set
val_months = sorted(demand['InvoiceMonth'].unique())[-2:]
val = demand[demand['InvoiceMonth'].isin(val_months)].copy()
val = val[val['monthly_quantity'] > 0]

mae  = np.mean(np.abs(val['monthly_quantity'] - val['predicted_quantity']))
mape = np.mean(np.abs((val['monthly_quantity'] - val['predicted_quantity']) / (val['monthly_quantity'] + 1))) * 100

print('=' * 50)
print('DEMAND FORECASTING MODEL')
print('=' * 50)
print(f'Algorithm           : LightGBM (log-transformed target)')
print(f'Validation period   : {val_months}')
print(f'MAE                 : {mae:.1f} units/month')
print(f'MAPE                : {mape:.1f}%')
print(f'Note                : MAPE inflated by low-volume products')

print()
print('=' * 50)
print('ELASTICITY MODEL')
print('=' * 50)
print(f'Method              : Log-log OLS regression per product')
print(f'Products analysed   : {len(elast):,}')
print(f'Statistically sig.  : {(elast["p_value"] < 0.05).sum()} (p < 0.05)')
print(f'Medium confidence   : {(elast["p_value"] < 0.20).sum()} (p < 0.20)')
print(f'Limitation          : 13 months data limits significance for fixed-price products')
print()
print(elast['category'].value_counts().to_string())

## 4. Pricing recommendations summary

In [ ]:
increase = recs[recs['action'].str.startswith('increase price')]
total_affected_revenue = increase['total_revenue'].sum()
estimated_uplift = (increase['total_revenue'] * increase['price_change_pct'] / 100).sum()

print('=' * 50)
print('PRICING RECOMMENDATIONS')
print('=' * 50)
print(f'Products analysed         : {len(recs):,}')
print(f'Price increases recommended: {len(increase)}')
print(f'  - Statistical evidence   : {(increase["confidence"] == "medium").sum()}')
print(f'  - Rule-based (inelastic) : {(increase["confidence"] == "rule-based").sum()}')
print(f'Average recommended increase: {increase["price_change_pct"].mean():.1f}%')
print(f'Revenue base of affected SKUs: £{total_affected_revenue:,.0f}')
print(f'Estimated annual uplift     : £{estimated_uplift * 12:,.0f}')
print()
print('Action breakdown:')
print(recs['action'].value_counts().to_string())

## 5. Full pipeline dashboard

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Monthly revenue trend
ax1 = fig.add_subplot(gs[0, :])
rev_trend = tx.groupby('InvoiceMonth')['Revenue'].sum()
ax1.plot(rev_trend.index, rev_trend.values, marker='o', color='steelblue', linewidth=2)
ax1.fill_between(rev_trend.index, rev_trend.values, alpha=0.15, color='steelblue')
ax1.set_title('Monthly revenue trend', fontsize=11)
ax1.set_ylabel('Revenue (£)')
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

# 2. Elasticity distribution
ax2 = fig.add_subplot(gs[1, 0])
elast['category'].value_counts().plot(kind='bar', ax=ax2, color=['#e74c3c','#95a5a6','#e67e22','#2ecc71'], edgecolor='white')
ax2.set_title('Products by elasticity', fontsize=10)
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=30, labelsize=8)

# 3. Demand forecast scatter
ax3 = fig.add_subplot(gs[1, 1])
sample = demand.sample(min(1000, len(demand)), random_state=42)
ax3.scatter(sample['monthly_quantity'], sample['predicted_quantity'], alpha=0.3, s=8, color='steelblue')
mx = max(sample['monthly_quantity'].max(), sample['predicted_quantity'].max())
ax3.plot([0, mx], [0, mx], 'r--', linewidth=1)
ax3.set_title('Demand forecast accuracy', fontsize=10)
ax3.set_xlabel('Actual')
ax3.set_ylabel('Predicted')

# 4. Pricing actions breakdown
ax4 = fig.add_subplot(gs[1, 2])
action_counts = recs['action'].value_counts()
colors = ['#2ecc71' if 'increase' in a else '#e74c3c' if 'sensitive' in a
          else '#3498db' if 'elastic' in a else '#95a5a6' for a in action_counts.index]
ax4.pie(action_counts.values, labels=None, colors=colors, startangle=90,
        autopct='%1.0f%%', pctdistance=0.75, textprops={'fontsize': 7})
ax4.set_title('Pricing actions', fontsize=10)

# 5. Top 10 opportunities
ax5 = fig.add_subplot(gs[2, :])
top_ops = increase.nlargest(10, 'total_revenue')
bars = ax5.barh(top_ops['description'].str[:35], top_ops['total_revenue'], color='steelblue')
ax5.set_xlabel('Total revenue (£)')
ax5.set_title('Top 10 pricing opportunities by revenue', fontsize=11)
for bar, chg in zip(bars, top_ops['price_change_pct']):
    ax5.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f'+{chg:.1f}%', va='center', fontsize=8, color='green')
ax5.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

fig.suptitle('Retailers AI Pricing — Pipeline Summary Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../reports/pipeline_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved to reports/pipeline_summary.png')

## 6. Top 20 final recommendations

In [ ]:
top20 = increase.nlargest(20, 'total_revenue')[[
    'description','current_price','recommended_price',
    'price_change_pct','elasticity_category','confidence','total_revenue'
]].copy()
top20.columns = ['Product','Current £','Recommended £','Change %','Category','Confidence','Revenue £']
top20['Revenue £'] = top20['Revenue £'].apply(lambda x: f'£{x:,.0f}')
print('=== TOP 20 PRICING RECOMMENDATIONS ===')
print(top20.to_string(index=False))

## 7. Next steps

| Step | Description | Priority |
|------|-------------|----------|
| A/B test | Run price changes on 5 products, measure demand response | High |
| More data | Add 2+ years of data to improve elasticity significance | High |
| Competitor prices | Scrape competitor pricing for benchmark | Medium |
| FastAPI | Expose recommendations via `/price/{sku}` endpoint | Medium |
| Dashboard | Build React frontend to visualise recommendations | Medium |
| Retrain schedule | Monthly retrain as new transactions come in | Low |
